# 7B on Kaggle T4s: tensor parallelism vs INT4 AWQ

Question: a 7B model in fp16 (15.2 GB of weights) does not fit on one 16 GB T4. Two ways out:

1. **Tensor parallelism (TP=2):** split every layer across both T4s. Same fp16 weights, two GPUs, all-reduce over PCIe (T4 has no NVLink).
2. **INT4 AWQ on one T4:** 4-bit weights (5.6 GB), one GPU, some quality risk.

Same model family (`Qwen2.5-7B-Instruct` vs its official AWQ build). Each config gets the same latency sweep and the same 100 GSM8K questions.

In [ ]:
# 1. Install vLLM and check hardware and disk
!pip install -q vllm 2>&1 | tail -2
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!nvidia-smi topo -m
!df -h /root /kaggle/working | tail -2

In [ ]:
# 2. Helpers and sizing math
import json, os, re, shutil, subprocess, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

MAX_LEN, GPU_UTIL = 4096, 0.90
VLLM_VISIBLE_GIB = 14.56   # what vLLM saw on a T4 in the first run (nvidia-smi says 15.0)
TRANSIENT_GIB = 0.45       # activation peak measured in the first run
CONFIGS = [
    {"name": "7b-fp16-tp2", "model": "Qwen/Qwen2.5-7B-Instruct", "tp": 2, "gpus": "0,1"},
    {"name": "7b-awq-1gpu", "model": "Qwen/Qwen2.5-7B-Instruct-AWQ", "tp": 1, "gpus": "0"},
]
hf = lambda path: json.load(urllib.request.urlopen(f"https://huggingface.co/{path}"))

for cfg in CONFIGS:
    c = hf(f"{cfg['model']}/resolve/main/config.json")
    w_gib = sum(f.get("size", 0) for f in hf(f"api/models/{cfg['model']}/tree/main") if f["path"].endswith(".safetensors")) / 2**30
    L, H, KVH = c["num_hidden_layers"], c["num_attention_heads"], c["num_key_value_heads"]
    kv = 2 * L * KVH * (c["hidden_size"] // H) * 2           # bytes per token, all GPUs together
    budget = VLLM_VISIBLE_GIB * GPU_UTIL                     # per GPU
    per_gpu_w = w_gib / cfg["tp"]
    free = budget - per_gpu_w - TRANSIENT_GIB
    cfg["pred_kv_tokens"] = int(free * 2**30 * cfg["tp"] / kv) if free > 0 else 0
    print(f"{cfg['name']}: weights {w_gib:.2f} GiB -> {per_gpu_w:.2f} GiB per GPU vs budget {budget:.2f} GiB per GPU")
    print(f"   KV per token {kv:,} B total, {kv//cfg['tp']:,} B per GPU (TP splits the {KVH} KV heads)")
    print(f"   predicted KV cache {cfg['pred_kv_tokens']:,} tokens = {cfg['pred_kv_tokens']/MAX_LEN:.0f} full {MAX_LEN}-token requests")
print(f"fp16 7B on ONE T4 would need {15.23e9/2**30:.2f} GiB for weights alone, more than the {VLLM_VISIBLE_GIB*GPU_UTIL:.2f} GiB budget.")

def gpu_used_mib():
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
    return [int(x) for x in out.split()]

def start_server(cfg):
    log = f"/kaggle/working/{cfg['name']}_server.log"
    srv = subprocess.Popen(["vllm", "serve", cfg["model"], "--dtype", "half", "--max-model-len", str(MAX_LEN),
                            "--gpu-memory-utilization", str(GPU_UTIL), "--tensor-parallel-size", str(cfg["tp"]), "--port", "8000"],
                           stdout=open(log, "w"), stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=cfg["gpus"]))
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen("http://localhost:8000/health"); break
        except Exception:
            if srv.poll() is not None:
                print(open(log).read()[-5000:]); raise SystemExit("server exited")
            if time.time() - t0 > 1800: raise SystemExit("server not ready after 30 min")
            time.sleep(3)
    print(f"{cfg['name']}: server ready after {time.time()-t0:.0f} s")
    text = open(log).read()
    for line in text.splitlines():
        if re.search(r"backend|quantiz|awq|model loading|kv cache size|maximum concurrency|all.?reduce|nccl|p2p", line, re.I):
            print("   ", line[line.find("]") + 1:].strip()[:200])
    grab = lambda pat: (re.findall(pat, text) or [None])[-1]
    cfg["load_gib"] = grab(r"Model loading took ([\d.]+) GiB")
    cfg["kv_tokens"] = grab(r"GPU KV cache size: ([\d,]+) tokens")
    return srv

def stop_server(srv):
    srv.terminate()
    try: srv.wait(timeout=90)
    except subprocess.TimeoutExpired: srv.kill(); srv.wait()
    for _ in range(60):
        if max(gpu_used_mib()) < 600: break
        time.sleep(2)
    print("server stopped, GPU MiB used:", gpu_used_mib())

def chat(model, prompt, max_tokens=512):
    body = json.dumps({"model": model, "messages": [{"role": "user", "content": prompt}],
                       "temperature": 0, "max_tokens": max_tokens}).encode()
    req = urllib.request.Request("http://localhost:8000/v1/chat/completions", body, {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=900))["choices"][0]["message"]["content"]

def sweep(cfg):
    chat(cfg["model"], "Say hi.", 8)  # warm-up, so the first measured request is not a cold start
    rows = []
    for c in [1, 4, 16, 64]:
        name = f"{cfg['name']}_c{c}.json"
        r = subprocess.run(["vllm", "bench", "serve", "--backend", "vllm", "--base-url", "http://localhost:8000",
                            "--model", cfg["model"], "--dataset-name", "random", "--random-input-len", "512",
                            "--random-output-len", "128", "--ignore-eos", "--num-prompts", str(max(16, 4 * c)),
                            "--max-concurrency", str(c), "--seed", "0", "--save-result",
                            "--result-dir", "/kaggle/working", "--result-filename", name], capture_output=True, text=True)
        if r.returncode:
            print(r.stdout[-3000:], r.stderr[-3000:]); raise SystemExit("bench failed")
        d = json.load(open(f"/kaggle/working/{name}"))
        rows.append({"config": cfg["name"], "concurrency": c, "TTFT median ms": round(d["median_ttft_ms"]),
                     "TTFT p99 ms": round(d["p99_ttft_ms"]), "TPOT median ms": round(d["median_tpot_ms"], 1),
                     "output tok/s": round(d["output_throughput"]), "requests/s": round(d["request_throughput"], 2)})
        print(rows[-1])
    return rows

GSM = pd.read_parquet("https://huggingface.co/datasets/openai/gsm8k/resolve/main/main/test-00000-of-00001.parquet").head(100)
GSM["gold"] = GSM.answer.str.extract(r"####\s*(-?[\d,.]+)")[0].str.replace(",", "")

def last_number(text):
    nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", text.split("Answer:")[-1])
    return nums[-1].replace(",", "") if nums else None

def gsm8k(cfg):
    t0 = time.time()
    ask = lambda q: chat(cfg["model"], q + "\nSolve it step by step. End with a final line 'Answer: <number>'.")
    with ThreadPoolExecutor(32) as ex:
        outs = list(ex.map(ask, GSM.question))
    ok = 0
    for out, gold in zip(outs, GSM.gold):
        pred = last_number(out)
        try: ok += pred is not None and abs(float(pred) - float(gold)) < 1e-6
        except ValueError: pass
    cfg["gsm8k"] = ok / len(GSM)
    print(f"{cfg['name']}: GSM8K {ok}/{len(GSM)} correct in {time.time()-t0:.0f} s")
    print("sample answer:", outs[0][-300:])

ALL_ROWS = []

In [ ]:
# 3. Config A: fp16 7B split across both T4s (TP=2)
cfg = CONFIGS[0]
srv = start_server(cfg)
print("GPU MiB used at idle:", gpu_used_mib())
ALL_ROWS += sweep(cfg)
gsm8k(cfg)
stop_server(srv)
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct"), ignore_errors=True)  # free 15 GB of disk

In [ ]:
# 4. Config B: INT4 AWQ 7B on one T4
cfg = CONFIGS[1]
srv = start_server(cfg)
print("GPU MiB used at idle:", gpu_used_mib())
ALL_ROWS += sweep(cfg)
gsm8k(cfg)
stop_server(srv)

In [ ]:
# 5. Side by side
df = pd.DataFrame(ALL_ROWS); df.to_csv("/kaggle/working/tp2_vs_awq_sweep.csv", index=False)
print(df.to_markdown(index=False))
summary = pd.DataFrame([{k: c.get(k) for k in ["name", "tp", "pred_kv_tokens", "kv_tokens", "load_gib", "gsm8k"]} for c in CONFIGS])
print(summary.to_markdown(index=False))
json.dump(CONFIGS, open("/kaggle/working/tp2_vs_awq_summary.json", "w"), indent=1)

## What to look for

- **Memory:** "Model loading took" per GPU. TP=2 halves the weights per GPU; AWQ cuts them about 2.7x (embeddings and lm_head stay fp16).
- **Latency at low load (TPOT at concurrency 1):** decode is bandwidth bound. TP=2 reads half the weights per GPU but pays an all-reduce per layer over PCIe. AWQ reads 4-bit weights but must dequantize them.
- **Throughput at high load:** TP=2 has two GPUs of compute and a bigger KV cache. AWQ has one GPU.
- **Quality:** GSM8K accuracy of AWQ against fp16 on the same 100 questions.